# Iris Flower Classification
OIBSIP · Data Science · Task 1

**Goal:** classify an iris flower as *Setosa*, *Versicolor* or *Virginica* from its four measurements.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 1. Load the data
The Iris dataset ships with scikit-learn, so no download is needed.

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
df = df.drop(columns="target")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nNull values:\n", df.isnull().sum())
df.describe()

In [ ]:
print(df["species"].value_counts())

### Pair plot
Each colour is a species. Look for features that separate the colours cleanly.

In [ ]:
sns.pairplot(df, hue="species", corner=True)
plt.show()

### Box plots per feature

In [ ]:
features = iris.feature_names
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, features):
    sns.boxplot(data=df, x="species", y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

### Feature discussion
**Petal length and petal width** separate the three species far better than sepal measurements. Setosa is completely separable on petals; Versicolor and Virginica overlap slightly, mostly on sepal width. We therefore expect petal features to carry most of the predictive power.

## 3. Train / test split (80 / 20)

In [ ]:
X = df[features]
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

## 4. Train and evaluate several classifiers

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "K-Nearest Neighbours": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    results[name] = acc
    print("=" * 60)
    print(name, "- accuracy:", round(acc, 4))
    print("\nConfusion matrix:\n", confusion_matrix(y_test, pred))
    print("\n", classification_report(y_test, pred))

In [ ]:
cm_fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, model) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test), labels=iris.target_names)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=iris.target_names, yticklabels=iris.target_names)
    ax.set_title(name)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 5. Compare models

In [ ]:
comparison = pd.Series(results, name="accuracy").sort_values(ascending=False)
print(comparison)
best = comparison.index[0]
print("\nBest model:", best)

## 6. Conclusion
Fill in after running: which model scored highest and why?

- All four models reach very high accuracy because Iris is small and almost linearly separable.
- Pick the best model from the table above. If several tie, prefer the simpler one (Logistic Regression or Decision Tree): it is easier to explain and less likely to overfit.
- With only 30 test samples, one wrong prediction changes accuracy by about 3%, so differences between models are not strong evidence.